[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Beanie Documents &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell. Run it first. Every cell below uses `await` at the top
level, which works because a notebook cell is already inside an event loop, and the tasks can be run
in any order.


In [1]:
import asyncio
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import beanie
import pydantic
import pymongo
from beanie import Document, init_beanie
from pymongo import AsyncMongoClient, IndexModel

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """A failure's real message, without the parts that change every run."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def first_problem(error):
    """One line out of a pydantic ValidationError, which is otherwise several paragraphs."""
    problem = error.errors()[0]
    return f"{'.'.join(str(part) for part in problem['loc'])}: {problem['msg']}"


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())
print("beanie", beanie.__version__, "| pydantic", pydantic.__version__)


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500
beanie 2.2.0 | pydantic 2.13.5


**1.** A model, and a database behind it.


In [2]:
class Part(Document):
    code: str
    name: str
    weight: float

    class Settings:
        name = "parts"


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Part])
await Part.delete_all()

print("collection:", Part.get_pymongo_collection().name)
print("fields:", list(Part.model_fields))


collection: parts
fields: ['id', 'revision_id', 'code', 'name', 'weight']


`id` is in the field list without being declared, because `Document` adds it. `Settings.name` is
what stops the collection being called `part`.


**2.** Before and after the insert.


In [3]:
part = Part(code="P-1", name="a bracket", weight=0.4)
print("before:", part.id)

await part.insert()
print("after: ", type(part.id).__name__)
print("and the object you already had now knows it:", part.id is not None)


before: None
after:  PydanticObjectId
and the object you already had now knows it: True


`insert` fills the `id` in on the object you passed, the same way `insert_one` filled in the `_id`
of a dictionary in **Collections and Documents**.


**3.** Several, then a question.


In [4]:
await Part.delete_all()
await Part.insert_many([
    Part(code="P-1", name="a bracket", weight=0.4),
    Part(code="P-2", name="a bolt", weight=0.05),
    Part(code="P-3", name="a plate", weight=2.5),
])

heavy = await Part.find(Part.weight > 0.3).sort(Part.code).to_list()
print("over 0.3 kg:", [(p.code, p.weight) for p in heavy])
print("count:", await Part.find(Part.weight > 0.3).count())


over 0.3 kg: [('P-1', 0.4), ('P-3', 2.5)]
count: 2


`find` builds the query and `to_list` runs it. The sort is on a model field, so a typo there would
be caught too.


**4.** Two typos, two outcomes.


In [5]:
try:
    await Part.find(Part.wieght > 1).to_list()
except AttributeError as error:
    print("through the model:", error)

quiet = await Part.find({"wieght": {"$gt": 1}}).to_list()
print("as a dictionary:  ", len(quiet), "documents, no error at all")


through the model: wieght
as a dictionary:   0 documents, no error at all


The model knows its fields and a raw dictionary does not. That is the whole reason to prefer the
expression form even though both are accepted.


**5.** What the model will not build.


In [6]:
for description, fields in (("weight as a word", {"code": "P-9", "name": "x", "weight": "heavy"}),
                            ("no name at all", {"code": "P-9", "weight": 1.0})):
    try:
        Part(**fields)
        print(f"  {description:18} accepted")
    except pydantic.ValidationError as error:
        print(f"  {description:18} {first_problem(error)}")


  weight as a word   weight: Input should be a valid number, unable to parse string as a number
  no name at all     name: Field required


Both refused in Python, before anything was sent. `first_problem` is there because a
`ValidationError` prints several paragraphs and a documentation link, and one line is enough here.


**6.** An index the model asks for.


In [7]:
class Coded(Document):
    code: str

    class Settings:
        name = "coded_parts"
        indexes = [IndexModel([("code", 1)], name="code_unique", unique=True)]


await client.get_default_database().coded_parts.drop()
await init_beanie(database=client.get_default_database(), document_models=[Coded])

print("built:", [index["name"] async for index
                 in await Coded.get_pymongo_collection().list_indexes()])

await Coded(code="C-1").insert()
try:
    await Coded(code="C-1").insert()
except pymongo.errors.DuplicateKeyError as error:
    print("and enforced:", failed(error))

await client.get_default_database().coded_parts.drop()
await client.close()


built: ['_id_', 'code_unique']
and enforced: DuplicateKeyError: E11000 duplicate key error collection: shop.coded_parts index: code_unique dup key: { code: "C-1" }


`init_beanie` created the index, so the constraint exists from the moment the program starts rather
than whenever somebody remembers to run a script.


---

&#8592; **Back to:** [Beanie Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/11-beanie-documents.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
